## XGBoost

In [1]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV   

train = pd.read_csv('../data/bot_identification/train.csv', sep=';')
test = pd.read_csv('../data/bot_identification/test.csv', sep=';')
val = pd.read_csv('../data/bot_identification/val.csv', sep=';')

encoder = OneHotEncoder()
x_train = train['text']
x_test = test['text']
x_val = val['text']

y_train = pd.DataFrame(train['account.type'])
y_test = pd.DataFrame(test['account.type'])
y_val = pd.DataFrame(val['account.type'])
y_train_matrix = encoder.fit_transform(y_train).toarray().tolist()
y_test_matrix = encoder.transform(y_test).toarray().tolist()
y_val_matrix = encoder.transform(y_val).toarray().tolist()

def get_single_label(encoded):
    y_labels = []
    for isBot, isntBot in encoded:
        y_labels.append(int(isBot))
    return y_labels
    
y_train_encoded = get_single_label(y_train_matrix)
y_test_encoded = get_single_label(y_test_matrix)
y_val_encoded = get_single_label(y_val_matrix)

In [3]:
# =========================================================
# Safe TF-IDF + XGBoost RandomizedSearchCV for 40-core node
# =========================================================

import os

# Put these BEFORE importing numpy / sklearn / xgboost when possible.
# If using Jupyter, restart the kernel before running this cell.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import json
import gc
import numpy as np
import joblib

from threadpoolctl import threadpool_limits

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, f1_score, make_scorer

from xgboost import XGBClassifier


# =========================================================
# Helper for JSON serialization
# =========================================================

def make_json_serializable(obj):
    """Convert NumPy/scikit-learn objects into JSON-safe Python objects."""
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, tuple):
        return list(obj)
    return str(obj)


# =========================================================
# Pipeline
# =========================================================

pipeline = Pipeline([
    (
        "features",
        FeatureUnion(
            transformer_list=[
                (
                    "word_tfidf",
                    TfidfVectorizer(
                        analyzer="word",
                        lowercase=True,
                        dtype=np.float32
                    )
                ),
                (
                    "char_tfidf",
                    TfidfVectorizer(
                        analyzer="char_wb",
                        lowercase=True,
                        dtype=np.float32
                    )
                )
            ],

            # Keep this at 1 because RandomizedSearchCV is parallelized.
            n_jobs=1
        )
    ),
    (
        "xgb",
        XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,

            # Safer 40-core strategy:
            # 5 CV jobs × 8 XGBoost threads ≈ 40 threads.
            n_jobs=8
        )
    )
])


# =========================================================
# Safer reduced parameter search space
# =========================================================

param_dist = {
    # -----------------------------
    # Word TF-IDF
    # -----------------------------
    "features__word_tfidf__ngram_range": [
        (1, 1),
        (1, 2)
    ],
    "features__word_tfidf__max_features": [
        20000,
        50000
    ],
    "features__word_tfidf__min_df": [
        2,
        3,
        5
    ],
    "features__word_tfidf__max_df": [
        0.90,
        0.95,
        1.0
    ],
    "features__word_tfidf__sublinear_tf": [
        True
    ],

    # -----------------------------
    # Character TF-IDF
    # -----------------------------
    "features__char_tfidf__ngram_range": [
        (3, 5),
        (4, 6)
    ],
    "features__char_tfidf__max_features": [
        20000,
        50000
    ],
    "features__char_tfidf__min_df": [
        2,
        3,
        5
    ],
    "features__char_tfidf__max_df": [
        0.90,
        0.95,
        1.0
    ],
    "features__char_tfidf__sublinear_tf": [
        True
    ],

    # -----------------------------
    # Word vs character weighting
    # -----------------------------
    "features__transformer_weights": [
        {"word_tfidf": 1.0, "char_tfidf": 1.0},
        {"word_tfidf": 1.0, "char_tfidf": 0.5},
        {"word_tfidf": 0.75, "char_tfidf": 1.0},
        {"word_tfidf": 1.25, "char_tfidf": 1.0},
        {"word_tfidf": 1.0, "char_tfidf": 1.25}
    ],

    # -----------------------------
    # XGBoost
    # -----------------------------
    "xgb__n_estimators": [
        300,
        500,
        800
    ],
    "xgb__max_depth": [
        3,
        4,
        5,
        6,
        8
    ],
    "xgb__learning_rate": [
        0.01,
        0.03,
        0.035,
        0.05,
        0.1
    ],
    "xgb__subsample": [
        0.7,
        0.8,
        0.85,
        0.9
    ],
    "xgb__colsample_bytree": [
        0.55,
        0.6,
        0.7,
        0.8
    ],
    "xgb__gamma": [
        0,
        0.1,
        0.3,
        0.5
    ],
    "xgb__min_child_weight": [
        1,
        2,
        3,
        5
    ],
    "xgb__reg_alpha": [
        0,
        0.01,
        0.1,
        0.3,
        1
    ],
    "xgb__reg_lambda": [
        1,
        5,
        10,
        20
    ],
    "xgb__scale_pos_weight": [
        1,
        2,
        2.5,
        5
    ],

    # Helps control memory for hist tree method
    "xgb__max_bin": [
        64,
        128,
        256
    ]
}


# =========================================================
# Cross-validation and scoring
# =========================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

f1_scorer = make_scorer(
    f1_score,
    pos_label=1
)


# =========================================================
# Randomized search
# =========================================================

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,

    # 50 iterations × 5 folds = 250 total fits
    # Safer than 100 iterations while debugging cluster memory.
    n_iter=50,

    scoring=f1_scorer,
    cv=cv,

    # Safer 40-core strategy:
    # 5 parallel CV jobs × 8 XGBoost threads = about 40 total threads.
    # This avoids launching 40 huge TF-IDF/XGBoost jobs at once.
    n_jobs=5,
    pre_dispatch=5,

    verbose=3,
    random_state=42,
    refit=True,
    return_train_score=False,

    # Use "raise" while debugging. Change to np.nan for unattended long runs.
    error_score="raise"
)


# =========================================================
# Fit search
# =========================================================

print("Starting RandomizedSearchCV...")

# Limit BLAS libraries to 1 thread.
# XGBoost still uses its own n_jobs=8 setting.
with threadpool_limits(limits=1, user_api="blas"):
    search.fit(x_train, y_train_encoded)

print("Search complete.")






Starting RandomizedSearchCV...
Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV 2/5] END features__char_tfidf__max_df=1.0, features__char_tfidf__max_features=50000, features__char_tfidf__min_df=5, features__char_tfidf__ngram_range=(4, 6), features__char_tfidf__sublinear_tf=True, features__transformer_weights={'word_tfidf': 1.0, 'char_tfidf': 1.0}, features__word_tfidf__max_df=1.0, features__word_tfidf__max_features=50000, features__word_tfidf__min_df=5, features__word_tfidf__ngram_range=(1, 1), features__word_tfidf__sublinear_tf=True, xgb__colsample_bytree=0.8, xgb__gamma=0.1, xgb__learning_rate=0.03, xgb__max_bin=256, xgb__max_depth=8, xgb__min_child_weight=3, xgb__n_estimators=500, xgb__reg_alpha=0.3, xgb__reg_lambda=20, xgb__scale_pos_weight=1, xgb__subsample=0.9;, score=0.816 total time= 6.9min
[CV 3/5] END features__char_tfidf__max_df=0.95, features__char_tfidf__max_features=50000, features__char_tfidf__min_df=3, features__char_tfidf__ngram_range=(4, 6), features_

NameError: name 'X_test' is not defined

[CV 4/5] END features__char_tfidf__max_df=1.0, features__char_tfidf__max_features=50000, features__char_tfidf__min_df=5, features__char_tfidf__ngram_range=(4, 6), features__char_tfidf__sublinear_tf=True, features__transformer_weights={'word_tfidf': 0.75, 'char_tfidf': 1.0}, features__word_tfidf__max_df=0.9, features__word_tfidf__max_features=20000, features__word_tfidf__min_df=5, features__word_tfidf__ngram_range=(1, 2), features__word_tfidf__sublinear_tf=True, xgb__colsample_bytree=0.6, xgb__gamma=0.5, xgb__learning_rate=0.035, xgb__max_bin=256, xgb__max_depth=3, xgb__min_child_weight=5, xgb__n_estimators=800, xgb__reg_alpha=0.3, xgb__reg_lambda=5, xgb__scale_pos_weight=1, xgb__subsample=0.8;, score=0.821 total time= 1.9min
[CV 4/5] END features__char_tfidf__max_df=0.9, features__char_tfidf__max_features=50000, features__char_tfidf__min_df=2, features__char_tfidf__ngram_range=(4, 6), features__char_tfidf__sublinear_tf=True, features__transformer_weights={'word_tfidf': 0.75, 'char_tfid

In [5]:
y_pred = best_model.predict(x_test)

report_dict = classification_report(
    y_test_encoded,
    y_pred,
    output_dict=True
)

report_text = classification_report(
    y_test_encoded,
    y_pred
)

results = {
    "best_cv_f1_score": search.best_score_,
    "best_parameters": search.best_params_,
    "classification_report": report_dict
}


# =========================================================
# Save results as JSON
# =========================================================

results_filename = "xgb_search_results.json"

with open(results_filename, "w") as f:
    json.dump(
        results,
        f,
        indent=4,
        default=make_json_serializable
    )

print(f"Saved results to {results_filename}")


# =========================================================
# Print summary
# =========================================================

print("\nBest CV F1 score:")
print(search.best_score_)

print("\nBest parameters:")
print(search.best_params_)

print("\nTest set classification report:")
print(report_text)

# =========================================================
# Save best model
# =========================================================

best_model = search.best_estimator_

model_filename = "big_grid_model_40core_safe.joblib"
joblib.dump(best_model, model_filename, compress=3)

print(f"Saved best model to {model_filename}")


# =========================================================
# Evaluate on test set
# =========================================================

# =========================================================
# Clean up memory
# =========================================================

gc.collect()

Saved results to xgb_search_results.json

Best CV F1 score:
0.8552033566706131

Best parameters:
{'xgb__subsample': 0.9, 'xgb__scale_pos_weight': 2.5, 'xgb__reg_lambda': 10, 'xgb__reg_alpha': 0.1, 'xgb__n_estimators': 500, 'xgb__min_child_weight': 3, 'xgb__max_depth': 6, 'xgb__max_bin': 64, 'xgb__learning_rate': 0.1, 'xgb__gamma': 0.1, 'xgb__colsample_bytree': 0.7, 'features__word_tfidf__sublinear_tf': True, 'features__word_tfidf__ngram_range': (1, 1), 'features__word_tfidf__min_df': 2, 'features__word_tfidf__max_features': 20000, 'features__word_tfidf__max_df': 1.0, 'features__transformer_weights': {'word_tfidf': 1.0, 'char_tfidf': 0.5}, 'features__char_tfidf__sublinear_tf': True, 'features__char_tfidf__ngram_range': (3, 5), 'features__char_tfidf__min_df': 2, 'features__char_tfidf__max_features': 20000, 'features__char_tfidf__max_df': 0.95}

Test set classification report:
              precision    recall  f1-score   support

           0       0.95      0.72      0.82      1278
    

9

In [ ]:
# from xgboost import XGBClassifier

# xgbclf = XGBClassifier()

# param_dist = {
#     "n_estimators": [100, 200, 300, 500, 800, 1200],
#     "max_depth": [3, 4, 5, 6, 8, 10, 13],
#     "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2, .035],
#     "subsample": [0.6, 0.7, 0.8, 0.9, .85, 1.0],
#     "colsample_bytree": [0.5, .55, 0.6, 0.7, 0.8, 1.0],
#     "gamma": [0, 0.1, 0.3, 0.5, 1],
#     "min_child_weight": [1, 2, 3, 5, 7, 10],
#     "reg_alpha": [0, 0.01, 0.1, .3, 1, 10],
#     "reg_lambda": [0, 0.1, 1, 5, 10, 20],
#     "scale_pos_weight": [1, 2, 2.5, 5, 10]
# }

In [ ]:
# rand_search_cv_xgb = RandomizedSearchCV(xgbclf, param_dist, n_jobs=-1, scoring=['accuracy','precision', 'recall', 'f1', 'roc_auc'], refit='f1', verbose=3)
# rand_search_cv_xgb.fit(x_train_vectorized, y_train_encoded)

In [ ]:
# rand_search_cv_xgb.best_params_

In [ ]:
# from sklearn.pipeline import Pipeline, FeatureUnion
# from sklearn.feature_extraction.text import TfidfVectorizer
# from xgboost import XGBClassifier

# # Get best params from completed RandomizedSearchCV
# best_xgb_params = rand_search_cv_xgb.best_params_

# features = FeatureUnion([
#     (
#         "word_tfidf",
#         TfidfVectorizer(
#             analyzer="word",
#             ngram_range=(1, 2),
#             max_features=50000,
#             min_df=2,
#             max_df=0.95,
#             sublinear_tf=True,
#             lowercase=True
#         )
#     ),
#     (
#         "char_tfidf",
#         TfidfVectorizer(
#             analyzer="char_wb",
#             ngram_range=(3, 5),
#             max_features=50000,
#             min_df=2,
#             sublinear_tf=True,
#             lowercase=True
#         )
#     )
# ])

# xgb = XGBClassifier(
#     **best_xgb_params,
#     eval_metric="logloss",
#     random_state=42,
#     n_jobs=-1
# )

# pipeline = Pipeline([
#     ("features", features),
#     ("xgb", xgb)
# ])

In [ ]:
pipeline = pipeline.fit(x_train, y_train_encoded)

In [ ]:
from sklearn.metrics import f1_score, classification_report

pipeline_pred = pipeline.predict(x_test)
f1 = f1_score(y_test_encoded, pipeline_pred)
print(f'{classification_report(y_test_encoded, pipeline_pred)}')

## saving xgb classifier and tfidf vectorizer used

In [ ]:
import joblib
from joblib import dump

filename = f"../models/bot_id_xgb_tree{f1}.joblib"

joblib.dump(pipeline, filename)